# Copywrite 2024, Duality Robotics

# This notebook references different informational pages.
# Create a free Falcon account now to access those pages.

# https://falcon.duality.ai/secure/documentation/getting-started



In [ ]:
# See how you can use images created from a FalconEditor simulation to train a
# YOLO_v8 model, then test the model on real world images.

# Read the comment instructions and run this colab notebook to see synthetic
# data work on real world images.

# FalconEditor allows users to easily create synthetic datasets tailored to
# their training needs. The FalconEDU subscription provides support (like this!)
# for training and testing.

# With FalconEDU, you have the instructions and software needed to
# create your own simulation, learn best practices for synthetic data and
# simualation, and collaborate with other learners and experts.


import os, sys
HOME = os.getcwd()
print(HOME)

In [ ]:
# Start with running a GPU enabled instance.  If this
# fails, go to the Edit menu, choose 'Notebook settings' and select a 'T4 GPU'
# under Hardware Accelerator.

# The GPU makes the training go MUCH faster, so it's worth setting up now.

# Not everyone has a GPU enabled computer, and even if you do, Colab still make
# the process easier.

# The FalconEDU subscription contains tips, tricks, and advice for making
# simulation using digital twins, synthetic data, and AI training easier, better,
# and faster!

!nvidia-smi

In [ ]:
# Now we install Ultralytics, the specific python version, and
# CUDA. These are all files needed for training.

# Using Colab prevents you from installing/uninstalling software
# versions, thereby bypassing potential conflicts. Our EDU support
# content helps you navigate these type of hurdles.

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output

import ultralytics
ultralytics.checks()

In [ ]:
# Now we mount (connect) the folder on Drive containing the training data.
# You will have to grant Colab access the drive connected to this Colab Notebook.
# After running the code below, the system will access for credentials - enter
# the ones for the particular Google account that controls the folder that
# contains the Falcon datasets.

# Note:  You may find that this does not work if the notebook and the Drive
# folder are owned by different Google accounts.

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive')

# Update SOURCE_PROJECT_ROOT to the shared folder path that contains the dataset.
# Example: SOURCE_PROJECT_ROOT = DRIVE_ROOT / 'Shared with me' / 'syntheticDataWorks_multiclass'
SOURCE_PROJECT_ROOT = DRIVE_ROOT / 'Multiple_object_detection'

DEST_PROJECT_FOLDER = 'syntheticDataWorks_multiclass'
DEST_PROJECT_ROOT = DRIVE_ROOT / DEST_PROJECT_FOLDER

if not SOURCE_PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f'Source project folder not found: {SOURCE_PROJECT_ROOT}\n'
        'Update SOURCE_PROJECT_ROOT to the folder that contains your shared dataset.'
    )

if SOURCE_PROJECT_ROOT.resolve() == DEST_PROJECT_ROOT.resolve():
    print('Source and destination are identical. Using the shared folder directly.')
else:
    if not DEST_PROJECT_ROOT.exists():
        import shutil
        print('Copying project from', SOURCE_PROJECT_ROOT, 'to', DEST_PROJECT_ROOT)
        shutil.copytree(SOURCE_PROJECT_ROOT, DEST_PROJECT_ROOT)
        print('Copy complete.')
    else:
        print('Destination already exists at', DEST_PROJECT_ROOT)

PROJECT_ROOT = DEST_PROJECT_ROOT
OUTPUT_DIR = PROJECT_ROOT / 'Output'
assert OUTPUT_DIR.exists(), f'Expected Output directory missing: {OUTPUT_DIR}'
print('Using project root:', PROJECT_ROOT)
print('Using output directory:', OUTPUT_DIR)

In [ ]:
# Change directory to the project folder in your mounted Drive.
# If your folder is not under MyDrive, update PROJECT_FOLDER above.

%cd {OUTPUT_DIR}

In [ ]:
# The project and dataset are already stored in Google Drive.
# This notebook will use the mounted Drive files directly, so there is no
# need to clone the GitHub repository here.

print('Project folder contents:')
!ls -la "{PROJECT_ROOT}"

In [ ]:
# This is a quick test to confirm that you've moved to the right folder and
# that the notebook can access the images.  You can hide the output after the
# test.

from enum import auto
from IPython.display import display
from ipywidgets import widgets, HBox, Layout
from ipywidgets import interact, interactive, fixed, interact_manual

imageA = widgets.Image(value=open(OUTPUT_DIR / 'train/images/000000000.png', 'rb').read(), layout = Layout(flex='0 1 auto', height='800px', min_height='40px', width='auto'))

display(imageA)

In [ ]:
# Navigate to the mounted Drive project's Output folder.

%cd {OUTPUT_DIR}

In [ ]:
# Now that we have access to the images, we can run the training script

# Please note that this will take a little while - 20-30 minutes.
# You may need to stay active on the page to avoid a timeout.
# You may notice that the output pauses at times, particuarly at the start.

# While this is running, read our docs explaining the training output:
# https://falcon.duality.ai/secure/documentation/ex3-colab-training-output

%run train.py

In [ ]:
# The above training outputs final metrics in this format:
# Model summary (fused): 168 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
#                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 5/5 [00:06<00:00,  1.39s/it]
#                   all        138        136      0.978      0.993      0.993       0.98

#      mAP50
# The  0.993  score is most important, but for this training that uses soley
# synthetic data, this high mAP50 score doesn't necesarily mean it works that well
# on real world images. We'll have to test this model next to see how well it performs
# in the real world.

# You can look at these graphs to check for any errors in the training.
from IPython.display import Image
results_path = OUTPUT_DIR / 'runs/train/exp/results.png'
if results_path.exists():
    display(Image(str(results_path)))
else:
    print(f"Results image not found at {results_path}. Training may not have completed successfully or the path is incorrect.")

# In depth information is at https://falcon.duality.ai/secure/documentation/ex3-colab-training-output
# On any graphs that say "loss", you should see a downward trend. If loss starts
# trending up, that means your model is getting worse and you should adjust your
# learning rate

# Precision and Recall should have an upward trend. If the data shoots up and flatlines,
# you are probably overfitting and need more variety in your dataset.

# The mAP50 trend is displayed. It should also have an upward trend, and if it
# starts trending back down, there is a problem

# Creating synthetic data in FalconEditor allows you to adjust your scenario if
# your training has any of these pitfalls, making getting effective training
# data easy, cheap, and accessible.

In [ ]:
# In order to see how the model performs in the physical world, we must test it
# on images from the physcial world.

# Run the test scipt by pressing "play" on this cell.
# This script tells the YOLO model to look at the images in the testImages
# folder and make predictions.

# If this isn't the first time you're run this training,
# the system will prompt you to select which model you'd like to validate. Just
# choose the highest number to test the latest.

# When this is finished, you can find the prediction images
# in ./syntheticDataWorks_multiclass/Output/predictions')
# You can find the training and testing metrics and graphs in
# ./syntheticDataWorks_multiclass/Output/runs/detect/. The train folder is from the training
# and the val folder is from the testing.
%run predict.py

In [ ]:
# Generate the final report and save output to report/generatedreport.d
!python ../report/report.py

In [ ]:
# Again, this process outputs this data:
# val: New cache created: /content/drive/MyDrive/Colab Notebooks/syntheticDataWorks_multiclass/testImages/labels.cache
#                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 11/11 [00:13<00:00,  1.22s/it]
#                   all        164        164      0.934       0.87      0.952      0.828

#      mAP50
# The  0.952  score shows that the training worked, and your model can detect the soup can in a real world setting. Congrats!

# Run this cell to see a grid of the prediction images!

from IPython.display import Image
Image(str(OUTPUT_DIR / 'runs/detect/val/val_batch2_pred.jpg'))


# You can see the full size, annotated predictions of the real world images here:
# ./syntheticDataWorks_multiclass/Output/predictions

# You can also see various output data in ./syntheticDataWorks_multiclass/Output/runs/detect/val
# You can read explanations of the output data (including a breakdown of
# the an mAP50 score meaning) here:
# https://falcon.duality.ai/secure/documentation/ex3-colab-testing-output


In [ ]:
# Distance analysis is removed because no additional distance data is provided.
# Continue with prediction, reporting, and visualization using the copied project data.